In [1]:
import pandas as pd

In [2]:
# ==========================================
# 1. BACA FILE EXCEL
# ==========================================
file_excel = 'data.xlsx'

In [3]:
# ==========================================
# 2. SOAL 1 & 2: DATA SISWA (Association Rule Preprocessing)
# ==========================================
df_siswa = pd.read_excel(file_excel, sheet_name='Siswa')

In [4]:
df_siswa.head()

,Tgl Lahir,Pekerjaan Wali,Pend,Thn msk,Thn lulus,UAS
0,6-Apr-93,Petani,SLTP,2000,2008,18.30
1,9-Aug-94,Petani,SD,2001,2008,16.45
2,25-May-95,Petani,SD,2001,2008,23.20
3,26-Jun-95,Petani,SD,2001,2008,18.75
4,14-Sep-95,Wiraswasta,SPG,2002,2008,23.10


In [5]:
# Hitung Usia Ketika Lulus
df_siswa['Thn Lahir'] = pd.to_datetime(df_siswa['Tgl Lahir'], format='%d-%b-%y').dt.year
df_siswa['Thn Lahir'] = df_siswa['Thn Lahir'].apply(lambda y: y - 100 if y > 2020 else y)
df_siswa['Usia Ketika Lulus'] = df_siswa['Thn lulus'] - df_siswa['Thn Lahir']

In [6]:
# One-Hot Encoding utuk atribut Pekerjaan Wali dan Pendidikan Terakhirnya
df_siswa_encoded = pd.get_dummies(df_siswa[['Pekerjaan Wali', 'Pend']], dtype=int)
df_siswa_final = pd.concat([df_siswa[['Usia Ketika Lulus']], df_siswa_encoded], axis=1)

print("--- Hasil Data Siswa ---")
df_siswa_final.head(10)

--- Hasil Data Siswa ---


,Usia Ketika Lulus,Pekerjaan Wali_ABRI,Pekerjaan Wali_PNS,Pekerjaan Wali_Pedagang,Pekerjaan Wali_Petani,Pekerjaan Wali_Wiraswasta,Pend_D III,Pend_D IV,Pend_SD,Pend_SLTA,Pend_SLTP,Pend_SMA,Pend_SPG,Pend_STM
0,15,0,0,0,1,0,0,0,0,0,1,0,0,0
1,14,0,0,0,1,0,0,0,1,0,0,0,0,0
2,13,0,0,0,1,0,0,0,1,0,0,0,0,0
3,13,0,0,0,1,0,0,0,1,0,0,0,0,0
4,13,0,0,0,0,1,0,0,0,0,0,0,1,0
5,13,0,0,0,0,1,0,0,0,0,1,0,0,0
6,13,0,0,0,1,0,0,0,1,0,0,0,0,0
7,12,0,0,0,0,1,0,0,0,0,0,1,0,0
8,12,0,0,0,1,0,0,0,1,0,0,0,0,0
9,12,0,0,0,0,1,0,0,0,0,0,0,0,1


In [7]:
# ==========================================
# 3. SOAL 3 & 4: DATA PELANGGAN (RFM Analysis)
# ==========================================
df_pelanggan = pd.read_excel(file_excel, sheet_name='Pelanggan')


In [8]:
# Konversi bulan ke Recency
recency_map = {'juni 2015': 0, 'Mei 2015': 1}
df_pelanggan['Recency_Val'] = df_pelanggan['Bulan'].map(recency_map)

In [9]:
# Agregasi RFM per Pelanggan
rfm_df = df_pelanggan.groupby('Nama').agg({
    'Recency_Val': 'min',
    'jumlah barang': 'sum',
    'Jumlah': 'sum'
}).rename(columns={'Recency_Val': 'Recency', 'jumlah barang': 'Frequency', 'Jumlah': 'Monetary'}).reset_index()

print("\n--- Hasil Data RFM Pelanggan ---")
rfm_df.head()


--- Hasil Data RFM Pelanggan ---


,Nama,Recency,Frequency,Monetary
0,Azkaqezanaza,1,1,160000
1,Bunda Dian,0,1,475000
2,Bunda Fatimah,1,1,130000
3,Dewy Felis Shoe,1,1,100000
4,Dewy Liza Gamis,1,1,160000


In [10]:
# ==========================================
# 4. SOAL 5 & 6: DATA PASIEN (Classification Preprocessing)
# ==========================================
df_pasien = pd.read_excel(file_excel, sheet_name='Pasien')
df_pasien.head()


,No.,No. Register,Nama Pasien,Alamat,Usia Ibu,Usia Kehamilan,HB,Jarak,Riwayat Penyakit,Riwayat Abortus,Gejala,Jenis Abortus,Tindakan
0,1,432514,Ny. Neni,cimanggu RT 01/01,20,10,> 11 gr%,> 2 th,tidak ada,tidak ada,pendarahan ringan,AB Iminens,Tidak Curret
1,2,440068,Ny. Ratini,cipari RT 05/03,33,5,< 11 gr%,< 2 th,tidak ada,tidak ada,"pendarahan,keramperut",AB Incomplit,Curret
2,3,432557,Ny. Tarmi,cilopadang RT 07/05,32,12,< 11 gr%,< 2 th,diabetes,tidak ada,"pendarahan, lemas",AB Incomplit,Curret
3,4,432812,Ny. Ida,sadahayu RT 03/03,29,6,> 11 gr%,> 2 th,tidak ada,tidak ada,keram rahim,AB Complit,Tidak Curret
4,5,432920,Ny.Susilowati,margasari RT 01/01,27,3,< 11 gr%,< 2 th,asma,1x,"flek, tegang payudara",AB Insipien,Curret


In [11]:
# Data Cleaning (Hapus identitas)
df_pasien_clean = df_pasien.drop(columns=['No.', 'No. Register', 'Nama Pasien', 'Alamat'])

In [12]:
# Data Transformation
df_pasien_clean['HB_Bin'] = df_pasien_clean['HB'].apply(lambda x: 1 if '> 11' in str(x) else 0)
df_pasien_clean['Jarak_Bin'] = df_pasien_clean['Jarak'].apply(lambda x: 1 if '> 2' in str(x) else 0)
df_pasien_clean['Riwayat_Abortus_Num'] = df_pasien_clean['Riwayat Abortus'].map({'tidak ada': 0, '1x': 1, '2x': 2})


In [13]:
print("\n--- Hasil Data Pasien Preprocessed ---")
df_pasien_clean.tail()


--- Hasil Data Pasien Preprocessed ---


,Usia Ibu,Usia Kehamilan,HB,Jarak,Riwayat Penyakit,Riwayat Abortus,Gejala,Jenis Abortus,Tindakan,HB_Bin,Jarak_Bin,Riwayat_Abortus_Num
5,37,9,< 11 gr%,< 2 th,diabetes,tidak ada,"pendarahan banyak, nyeri perut",AB Insipien,Curret,0,0,0
6,26,2,< 11 gr%,< 2 th,tbc,2x,"Perut kosong, kehamilan menghilang",Miss Abortion,Curret,0,0,2
7,24,4,> 11 gr%,> 2 th,toksoplasma,tidak ada,"Infeksi, pendarahan",AB Iminens,Tidak Curret,1,1,0
8,34,15,> 11 gr%,> 2 th,hipertensi,tidak ada,pendarahan,AB Complit,Tidak Curret,1,1,0
9,29,11,< 11 gr%,< 2 th,asma,1x,"pendarahan, keram perut",AB Incomplit,Curret,0,0,1


## POST TESTTTTT

1. Buat kolom baru bernama kategori UAS dengan ketentuan:
uas >= 22 tinggi
uas < 22 rendah

2. one hot encording pekerjaan wali dan kategori uas
3. 5 baris paertama hasil encoding

soal 2 data pelanggan ubah acuan patokan bulan terkini menjadi bulan agustus 2015
- 'juni 2015' -> Jarak 2 bulan
- mei 2015 jarak 3 bulan

hitung ulang tabel RFM dan tampilkan hasil nilai recency frequency serta monetary

soal 3 data pasien lakukan filtering pada dataframe df_pasien_clean yang telah diproses sebelumnya tampilkan baris data pasien yang memenuhi syarat:
- usia ibu diatas 30 tahun
- dan tidak memiliki riwayat abortus



1. Buat kolom baru bernama kategori UAS dengan ketentuan: uas >= 22 tinggi uas < 22 rendah

In [14]:
df_siswa['Kategori_UAS'] = df_siswa['UAS'].apply(lambda y: "Tinggi" if y >= 22 else "Rendah")

2. one hot encording pekerjaan wali dan kategori uas

In [15]:
df_siswa_encoded2 = pd.get_dummies(df_siswa[['Pekerjaan Wali', 'Kategori_UAS']], dtype=int)

- 5 baris paertama hasil encoding

In [16]:
df_siswa_encoded2.head(5)

,Pekerjaan Wali_ABRI,Pekerjaan Wali_PNS,Pekerjaan Wali_Pedagang,Pekerjaan Wali_Petani,Pekerjaan Wali_Wiraswasta,Kategori_UAS_Rendah,Kategori_UAS_Tinggi
0,0,0,0,1,0,1,0
1,0,0,0,1,0,1,0
2,0,0,0,1,0,0,1
3,0,0,0,1,0,1,0
4,0,0,0,0,1,0,1


soal 2 data pelanggan ubah acuan patokan bulan terkini menjadi bulan agustus 2015

In [26]:
# Konversi bulan ke Recency
recency_map = {'juni 2015': 2, 'Mei 2015': 3}
df_pelanggan['Recency_Val_August'] = df_pelanggan['Bulan'].map(recency_map)

In [27]:
# Agregasi RFM per Pelanggan
rfm_df2 = df_pelanggan.groupby('Nama').agg({
    'Recency_Val_August': 'min',
    'jumlah barang': 'sum',
    'Jumlah': 'sum'
}).rename(columns={'Recency_Val_August': 'Recency', 'jumlah barang': 'Frequency', 'Jumlah': 'Monetary'}).reset_index()

print("\n--- Hasil Data RFM Pelanggan ---")
rfm_df2.head()


--- Hasil Data RFM Pelanggan ---


,Nama,Recency,Frequency,Monetary
0,Azkaqezanaza,3,1,160000
1,Bunda Dian,2,1,475000
2,Bunda Fatimah,3,1,130000
3,Dewy Felis Shoe,3,1,100000
4,Dewy Liza Gamis,3,1,160000


soal 3 data pasien lakukan filtering pada dataframe df_pasien_clean yang telah diproses sebelumnya tampilkan baris data pasien yang memenuhi syarat:

usia ibu diatas 30 tahun
dan tidak memiliki riwayat abortus

In [25]:
df_pasien_clean[(df_pasien_clean['Usia Ibu'] > 30) & (df_pasien_clean['Riwayat_Abortus_Num'] == 0)]

,Usia Ibu,Usia Kehamilan,HB,Jarak,Riwayat Penyakit,Riwayat Abortus,Gejala,Jenis Abortus,Tindakan,HB_Bin,Jarak_Bin,Riwayat_Abortus_Num
1,33,5,< 11 gr%,< 2 th,tidak ada,tidak ada,"pendarahan,keramperut",AB Incomplit,Curret,0,0,0
2,32,12,< 11 gr%,< 2 th,diabetes,tidak ada,"pendarahan, lemas",AB Incomplit,Curret,0,0,0
5,37,9,< 11 gr%,< 2 th,diabetes,tidak ada,"pendarahan banyak, nyeri perut",AB Insipien,Curret,0,0,0
8,34,15,> 11 gr%,> 2 th,hipertensi,tidak ada,pendarahan,AB Complit,Tidak Curret,1,1,0
